# 05 - Arvore de decisao: risco de turnoverArvore de decisao e um modelo que cria regras para prever uma categoria. Aqui a categoria e `desligou_12m`: 1 significa desligou, 0 significa nao desligou.Este notebook e didatico. Em uma empresa real, modelos de turnover exigem governanca, privacidade, validacao e revisao etica.

## Cuidado etico antes do codigoUma previsao de turnover nunca deve ser usada para punir uma pessoa. O uso responsavel e investigar condicoes de trabalho, carga, carreira, lideranca, apoio e oportunidades.

In [ ]:
# pathlib localiza arquivos.from pathlib import Path# pandas trabalha com tabelas.import pandas as pd# Ferramentas de preprocessamento e modelagem.from sklearn.compose import ColumnTransformerfrom sklearn.metrics import accuracy_score, classification_report, confusion_matrixfrom sklearn.model_selection import train_test_splitfrom sklearn.pipeline import Pipelinefrom sklearn.preprocessing import OneHotEncoderfrom sklearn.tree import DecisionTreeClassifier, export_text# Localizamos e abrimos a base.BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()colaboradores = pd.read_csv(BASE_DIR / "dados" / "colaboradores.csv")# Conferimos as primeiras linhas.colaboradores.head()

## 1. Escolher alvo e variaveis explicativasO alvo e `desligou_12m`. As variaveis explicativas sao sinais historicos como absenteismo, horas extras, tempo de empresa e performance.

In [ ]:
# Coluna que queremos prever.alvo = "desligou_12m"# Variaveis numericas usadas pelo modelo.variaveis_numericas = [    "idade",    "tempo_empresa_meses",    "salario_mensal",    "horas_treinamento_ano",    "absenteismo_dias_ano",    "horas_extras_mes",    "divergencias_ponto_mes",    "nota_performance",]# Variaveis categoricas precisam ser convertidas com OneHotEncoder.variaveis_categoricas = ["area", "nivel"]# Lista completa de variaveis.variaveis = variaveis_numericas + variaveis_categoricas# X contem as informacoes usadas para prever.X = colaboradores[variaveis]# y contem o resultado real.y = colaboradores[alvo]# Vemos a proporcao de cada classe.y.value_counts(normalize=True).rename("proporcao")

## 2. Criar pipeline e separar treino/testeUsamos pipeline para garantir que o preprocessamento e o modelo sejam aplicados juntos. A arvore tem profundidade limitada para ficar mais interpretavel.

In [ ]:
# Preprocessamento: categorias viram colunas numericas; numeros passam direto.preprocessamento = ColumnTransformer(    transformers=[        ("categorias", OneHotEncoder(handle_unknown="ignore"), variaveis_categoricas),        ("numeros", "passthrough", variaveis_numericas),    ])# Criamos uma arvore pequena.# max_depth=3 limita a quantidade de perguntas em sequencia.# min_samples_leaf=3 evita folhas com pouquissimos exemplos.modelo = Pipeline(    steps=[        ("preprocessamento", preprocessamento),        ("arvore", DecisionTreeClassifier(max_depth=3, min_samples_leaf=3, random_state=42)),    ])# stratify=y tenta manter a proporcao de desligou/nao desligou em treino e teste.X_treino, X_teste, y_treino, y_teste = train_test_split(    X,    y,    test_size=0.30,    random_state=42,    stratify=y,)# Treinamos o modelo.modelo.fit(X_treino, y_treino)# Geramos previsoes para o conjunto de teste.previsoes = modelo.predict(X_teste)

## 3. Avaliar resultadosAcuracia mostra o percentual geral de acertos. Matriz de confusao mostra onde o modelo acerta e erra. Em RH, os tipos de erro importam muito.

In [ ]:
# Acuracia geral.print(f"Acuracia: {accuracy_score(y_teste, previsoes):.2f}")# Matriz de confusao.# Linhas sao valores reais; colunas sao previsoes.print("Matriz de confusao:")print(confusion_matrix(y_teste, previsoes))# Relatorio com precisao, recall e f1-score.print("Relatorio de classificacao:")print(classification_report(y_teste, previsoes, target_names=["Nao desligou", "Desligou"], zero_division=0))

## 4. Ler regras da arvoreUma vantagem da arvore e que conseguimos imprimir regras. Isso ajuda na explicacao, mas nao elimina o cuidado etico.

In [ ]:
# Recuperamos o preprocessador treinado dentro do pipeline.preprocessador_treinado = modelo.named_steps["preprocessamento"]# Pegamos os nomes das colunas depois do OneHotEncoder.nomes_features = preprocessador_treinado.get_feature_names_out()# Recuperamos a arvore treinada.arvore = modelo.named_steps["arvore"]# Imprimimos as regras em texto.print(export_text(arvore, feature_names=list(nomes_features)))

## 5. Interpretar como RHUma interpretacao ruim seria: `a pessoa vai sair`.Uma interpretacao melhor seria: `no historico didatico, determinados sinais aparecem associados a maior taxa de desligamento; isso pode orientar investigacao sobre carga, apoio, lideranca, carreira e condicoes de trabalho`.

## 6. Exercicios1. Altere `max_depth` para 2 e compare as regras.2. Remova `salario_mensal` das variaveis e rode novamente.3. Observe se a matriz de confusao mudou.4. Escreva uma recomendacao de RH que nao seja punitiva.5. Escreva uma frase explicando por que este modelo nao deve ser usado para decidir desligamentos.